In [6]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
from tqdm import tqdm
import os
from pathlib import Path
import pickle


In [7]:
coordinates_path = Path("/Users/thomasbush/Downloads/multicam_video_2025-05-07T12_16_20_cropped-v2_20250701121021_triangulated_points_20250802-065459.h5")
arena_path = Path("/Users/thomasbush/Documents/Vault/Iurilli_lab/3d_tracking/3d-setup/tests/assets/arena_views_triangulated.h5")
cricket_boottom_path = Path("/Users/thomasbush/Downloads/multicam_video_2025-05-07T12_16_20_centralDLC_HrnetW48_cricket-bottomJul1shuffle1_detector_170_snapshot_066_full.pickle")

In [8]:
# load the coordinates:
data = xr.open_dataset(coordinates_path)
arena = xr.open_dataset(arena_path)
cricket_bottom = pickle.load(open(cricket_boottom_path, "rb"))
# get max, min value for each dimension:

In [9]:
# get max, min value for each dimension:
for dim in range(data.position.shape[1]):
    print(f"Dimension {dim}:")
    print(f"  Max value: {data.position[:, dim, ...].max().values}")
    print(f"  Min value: {data.position[:, dim, ...].min().values}")
    print(f"  Mean value: {data.position[:, dim, ...].mean().values}")
    print(f"  Median value: {np.median(data.position[:, dim, ...].values)}")
    print(f"  Standard deviation: {data.position[:, dim, ...].std().values}")

Dimension 0:
  Max value: 4161.337505112246
  Min value: -33203.9494573142
  Mean value: 11.884648066956201
  Median value: 82.2357387171931
  Standard deviation: 112.0987189609319
Dimension 1:
  Max value: 40286.782731647225
  Min value: -5924.840005945679
  Mean value: -52.35860607911988
  Median value: -113.62901059591513
  Standard deviation: 121.87383801397763
Dimension 2:
  Max value: 28204.471663744553
  Min value: 631.1914236775026
  Mean value: 990.9745054059777
  Median value: 986.384091922628
  Standard deviation: 31.483779832635516


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [123]:
# function to ge the velocity 
from typing import Any


def get_velocity(session:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Calculate velocity from position data
    
    Args:
        session (xr.Dataset): The input dataset containing position data
        time_slice (slice): The time slice to calculate velocity over
        
    Returns:
        np.ndarray: The velocity data
    """
    if time_slice:
        coordinates = session.position.isel(time=slice(0,time_slice)).values
    centroids = coordinates.mean(axis=2).squeeze()
    velocity = np.vstack([np.zeros((1, 3)), np.diff(centroids, axis=0)])
    velocity = np.linalg.norm(velocity, axis=1)
    return velocity

def get_accelleration(session:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Calculate accelleration from position data
    
    Args:
        session (xr.Dataset): The input dataset containing position data
        time_slice (slice): The time slice to calculate accelleration over
        
    Returns:
        np.ndarray: The accelleration data
    """
    if time_slice:
        coordinates = session.position.isel(time=slice(0,time_slice)).values
    centroids = coordinates.mean(axis=2).squeeze()
    displacement = np.vstack([np.zeros((1, 3)), np.diff(centroids, axis=0)])
    velocity = displacement 
    accelleration = np.vstack([np.zeros((1, 3)), np.diff(velocity, axis=0)]) 
    accelleration = np.linalg.norm(accelleration, axis=1)
    return accelleration

def get_head_rear(session:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Computes the verrtifical movement of the head as an heuristic of rear

    Args:
        session(xr.Dataset): input xrarray 
        time_slice: int max frame
    Returns:
        np.ndarray
    """
    rear = data["position"].sel(keypoints=["nose", "ear_lf", "ear_rt"], space="z").isel(time=slice(0, time_slice)).values.squeeze()
    return rear
def get_theta(session:xr.Dataset, time_slice:int, keypoints:tuple)->np.ndarray:
    """
    Computes the theta angle of the head in the 2D xy space
    
    Args:
        session(xr.Dataset): input xr.Dataset
        time_slice: int max frame
    Returns:
        The direction fo the head in the 2D xy space [-pi, pi]
    """
    keypoint_1, keypoint_2 = keypoints
    diff = data["position"].sel(keypoints=keypoint_1).values - data["position"].sel(keypoints=keypoint_2).values
    diff = diff[:time_slice]
    diff_norm = np.linalg.norm(diff, axis=1)
    v_x = diff[:, 0] / diff_norm
    v_y = diff[:, 1] / diff_norm
    theta_head = np.arctan2(v_y, v_x)
    return theta_head
    
def get_turning_rate(session:xr.Dataset, time_slice:int, keypoints:tuple)->np.ndarray:
    """
    Computes the turning rate of the head
    Args:
        session(xr.Dataset): input xr.Dataset
        time_slice: int max frame
        keypoints: tuple of keypoints
    Returns:
        np.ndarray: turning rate
    """
    theta_head = get_theta(session, time_slice, keypoints)
    dtheta = np.vstack([np.zeros((1,)), np.diff(theta_head, axis=0)])
    #wrapped to [-pi, pi]
    dtheta = np.mod(dtheta + np.pi, 2 * np.pi) - np.pi
    dtheta = dtheta 
    return dtheta
def get_yaw_offset(session:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Computes the yaw offset of the head
    Args:
        session(xr.Dataset): input xr.Dataset
        time_slice: int max frame
    Returns:
        np.ndarray: 0 (body and head are aligned), >0 (head is left of body), <0 (head is right of body)
    """
    theta_body = get_theta(session, time_slice, ("back_mid", "tailbase"))
    theta_head = get_theta(session, time_slice, ("nose", "tailbase"))
    delta_theta = theta_head - theta_body
    #wrap to [-pi, pi]
    delta_theta = np.mod(delta_theta + np.pi, 2 * np.pi) - np.pi
    return delta_theta

def get_pitch_angle(session:xr.Dataset, time_slice:int, keypoints:tuple)->np.ndarray:
    """
    Computes the pitch angle of the head

    Args:
        session(xr.Dataset): input xr.Dataset
        time_slice: int max frame
        keypoints: tuple of keypoints
    Returns:
        np.ndarray: pitch angle
        pitch = 0 when head and body are aligned, pitch > 0 when head is up, pitch < 0 when head is down
    """
    keypoint_1, keypoint_2 = keypoints
    if time_slice:
        v_nose = session.position.sel(keypoints=keypoint_1).values[:time_slice]
        v_tailbase = session.position.sel(keypoints=keypoint_2).values[:time_slice]
    else:
        v_nose = session.position.sel(keypoints=keypoint_1).values
        v_tailbase = session.position.sel(keypoints=keypoint_2).values
    v_body = v_nose - v_tailbase
    # compute horizontal (ground-plane) lenght 
    L_xy = np.linalg.norm(v_body[:, :2], axis=1)
    #compute pitch angle
    pitch_angle = np.arctan2(v_body[:, 2], L_xy)
    return pitch_angle
def get_velocity_components(session:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Computes the forward, sideways, and vertical velocity components

    Args:
        session(xr.Dataset): input xr.Dataset
        time_slice: int max frame
    Returns:
        np.ndarray: forward, sideways, and vertical velocity components
    """
    # 1. Centroid positions and 3D displacement (velocity vector per frame)
    centroids = session.position.isel(time=slice(0, 100)).values.mean(axis=2).squeeze()
    # centroids.shape -> (T, 3)

    displacement = np.vstack([np.zeros((1, 3)), np.diff(centroids, axis=0)])
    # displacement.shape -> (T, 3)

    # 2. Body direction: tailbase -> nose, unit vector per frame
    nose = session.position.sel(keypoints="nose").isel(time=slice(0, 100)).values
    tail = session.position.sel(keypoints="tailbase").isel(time=slice(0, 100)).values

    # nose, tail: (T, 3, 1) -> drop individuals dim
    nose = nose.squeeze(-1)   # (T, 3)
    tail = tail.squeeze(-1)   # (T, 3)

    body_vector = nose - tail           # (T, 3)
    body_vector_norm = np.linalg.norm(body_vector, axis=1)  # (T,)
    body_vector_unit = body_vector / body_vector_norm[:, None]  # (T, 3)

    # 3. Sideways axis from cross product with global z-axis
    z_axis = np.array([0.0, 0.0, 1.0])  # (3,)

    side_axis = np.cross(z_axis, body_vector_unit)      # (T, 3)
    side_axis_norm = np.linalg.norm(side_axis, axis=1)  # (T,)
    side_axis_unit = side_axis / side_axis_norm[:, None]  # (T, 3)

    # 4. Per-frame projections (dot products)
    # forward: along body_vector_unit
    forward_velocity = np.sum(displacement * body_vector_unit, axis=1)  # (T,)

    # sideways: along side_axis_unit
    side_velocity = np.sum(displacement * side_axis_unit, axis=1)       # (T,)

    # vertical: along z-axis (just the z component of displacement)
    vertical_velocity = displacement[:, 2]                               # (T,)
    # or: vertical_velocity = np.sum(displacement * z_axis, axis=1)
    return forward_velocity, side_velocity, vertical_velocity
def get_manipulation_index_paws(session:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Computes the manipulation index of the paws
    """
    v_lf = session.position.sel(keypoints="forepaw_lf").values.squeeze()[:time_slice]
    v_rt = session.position.sel(keypoints="forepaw_rt").values.squeeze()[:time_slice]
    body_speed = get_velocity(session, time_slice)
    lf_disp = np.vstack([np.zeros((1, 3)), np.diff(v_lf, axis=0)])
    rt_disp = np.vstack([np.zeros((1, 3)), np.diff(v_rt, axis=0)])
    lf_speed = np.linalg.norm(lf_disp, axis=1)
    rt_speed = np.linalg.norm(rt_disp, axis=1)
    manipulation_idx_lf = lf_speed / body_speed
    manipulation_idx_rt = rt_speed / body_speed
    return manipulation_idx_lf, manipulation_idx_rt

def get_freezing(session:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Computes the freezing of the animal
    """
    velocity = get_velocity(session, time_slice)
    threshold = np.percentile(velocity, 10)
    freezing = velocity < threshold
    return freezing
def get_curvature(session:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Computes the curvature of the path
    """
    turning_rate = get_turning_rate(session, time_slice,  keypoints=("nose", "tailbase"))
    velocity = get_velocity(session, time_slice)
    curvature = np.abs(turning_rate.squeeze()) / velocity + np.random.randn(1000) * 0.01
    return curvature
def get_position_centroid(session:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Computes the centroid of the position
    """
    centroid = session.position.isel(time=slice(0, time_slice)).values.mean(axis=2).squeeze()
    return centroid

def get_distance_to_walls(session:xr.Dataset, arena:xr.Dataset, time_slice:int)->np.ndarray:
    """
    Computes the distance to the walls
    """
    centroid_position = get_position_centroid(data, time_slice=time_slice)
    arena_2d = arena.position.sel(space=["x", "y"]).values.squeeze()
    x_min, x_max = arena_2d[:, 0].min(), arena_2d[:, 0].max()
    y_min, y_max = arena_2d[:, 1].min(), arena_2d[:, 1].max()

    # Use only 2D position from centroids (first two dimensions)
    centroid_2d = centroid_position[:, :2]  # shape: (frame, 2)

    # Compute distance to nearest wall for each frame
    dist_to_x_min = np.abs(x_min - centroid_2d[:, 0])
    dist_to_x_max = np.abs(x_max - centroid_2d[:, 0])
    dist_to_y_min = np.abs(y_min - centroid_2d[:, 1])
    dist_to_y_max = np.abs(y_max - centroid_2d[:, 1])

    # For each frame, find the minimum distance to any wall
    d_wall = np.minimum.reduce([dist_to_x_min, dist_to_x_max, dist_to_y_min, dist_to_y_max])  # shape: (frame,)
    return d_wall

def load_cricket_coordinates(pickle_path:Path)->np.ndarray:
    """
    Loads the coordinates of the cricket
    """
    data_cricket = pickle.load(open(pickle_path, "rb"))
    # we have a dict with "metadata" and one key for each frame:
    # we want to extract the coordinates for each frame and put them in an array:
    coordinates = []
    for frame in data_cricket.keys():
        #skip metadata
        if frame == "metadata":
            continue
        # get the coordinates:
        coordinates.append(data_cricket[frame]["coordinates"][0][0])
    return np.array(coordinates)
def convert_cricket_coordinates(cricket_coordinates:np.ndarray, arena_3d:xr.Dataset, arena_views:xr.Dataset)->np.ndarray:
    """
    Converts the cricket coordinates to arena floor coordinates
    """
    # Extract 4 corner correspondences
    arena_corners_3d = arena_3d["position"].sel(space=["x", "y"]).values.squeeze()[:, :4]  # shape: (2, 4) - [x, y] x 4 corners
    arena_corners_2d = arena_views["position"].sel(view="central").values.squeeze()[:, :4]  # shape: (2, 4) - [x, y] x 4 corners
        # For 3D arena coordinates (destination)
    arena_3d_homogeneous = np.vstack([arena_corners_3d, np.ones((1, 4))])  # shape: (3, 4) - each column is [x, y, 1]
    # For 2D image coordinates (source)
    arena_2d_homogeneous = np.vstack([arena_corners_2d, np.ones((1, 4))])  # shape: (3, 4) - each column is [x, y, 1]

    # Compute homography using Direct Linear Transform (DLT)
    # H maps from 2D image to 3D arena: [x', y', 1]^T = H @ [x, y, 1]^T
    # Using pseudo-inverse: H = arena_3d_homogeneous @ pinv(arena_2d_homogeneous)
    H = arena_3d_homogeneous @ np.linalg.pinv(arena_2d_homogeneous)

    # Apply homography to cricket coordinates
    # Prepare cricket coordinates in homogeneous form: each column is [x, y, 1]
    cricket_2d = cricket_coordinates.squeeze()  # shape: (N, 2)
    cricket_2d_homogeneous = np.vstack([cricket_2d.T, np.ones((1, cricket_2d.shape[0]))])  # shape: (3, N)

    # Transform: cricket_3d_homogeneous = H @ cricket_2d_homogeneous
    cricket_3d_homogeneous = H @ cricket_2d_homogeneous  # shape: (3, N)

    # Normalize by homogeneous coordinate (last row) to get [x, y, 1]
    # Divide each column by its third element (the homogeneous coordinate)
    cricket_3d_homogeneous = cricket_3d_homogeneous / cricket_3d_homogeneous[2, :]  # shape: (3, N)

    # Extract x, y coordinates (first two rows)
    cricket_converted = cricket_3d_homogeneous[:2, :].T  # shape: (N, 2)
    return cricket_converted
def get_distance_mouse_cricket(mouse_coordinates:np.ndarray, cricket_coordinates:np.ndarray)->np.ndarray:
    """
    Computes the distance between the mouse and the cricket
    """
    return np.linalg.norm(mouse_coordinates[:, :2] - cricket_coordinates, axis=1)


Remaining features to compute:
- Freezing, immobility
- Distance to walls and center
- path curvature
- gait: stride frequency and stride lenght for the forepaws

In [121]:
arena_3d = Path("/Users/thomasbush/Documents/Vault/Iurilli_lab/3d_tracking/data/newarena.h5")
arena_views = Path("/Users/thomasbush/Documents/Vault/Iurilli_lab/3d_tracking/3d-setup/tests/assets/arena_views.h5")
arena_3d = xr.open_dataset(arena_3d)
arena_views = xr.open_dataset(arena_views)

In [124]:
velocity = get_velocity(data, time_slice=1000)
acc = get_accelleration(data, time_slice=1000)
rear = get_head_rear(data, time_slice=1000)
theta_head = get_theta(data, time_slice=100, keypoints=("nose", "tailbase"))
turning_rate = get_turning_rate(data, time_slice=1000, keypoints=("nose", "tailbase"))
yaw_offset = get_yaw_offset(data, time_slice=1000)
pitch_angle = get_pitch_angle(data, time_slice=1000, keypoints=("nose", "tailbase"))
forward_velocity, side_velocity, vertical_velocity = get_velocity_components(data, time_slice=1000)
manipulation_idx_lf, manipulation_idx_rt = get_manipulation_index_paws(data, time_slice=1000)
freezing = get_freezing(data, time_slice=1000)
curvature = get_curvature(data, time_slice=1000)
centroid_position = get_position_centroid(data, time_slice=1000)
d_wall = get_distance_to_walls(data, arena, time_slice=1000)

cricket_bottom_coordinates_raw = load_cricket_coordinates(cricket_boottom_path)
cricket_converted = convert_cricket_coordinates(cricket_bottom_coordinates_raw, arena_3d, arena_views)
distance_mouse_cricket = get_distance_mouse_cricket(centroid_position, cricket_converted[:1000])

/var/folders/0m/4nz5dl3956v2_pcbzw2lwtwr0000gn/T/ipykernel_21748/1226303023.py:189: RuntimeWarning:

invalid value encountered in divide

/var/folders/0m/4nz5dl3956v2_pcbzw2lwtwr0000gn/T/ipykernel_21748/1226303023.py:190: RuntimeWarning:

invalid value encountered in divide

/var/folders/0m/4nz5dl3956v2_pcbzw2lwtwr0000gn/T/ipykernel_21748/1226303023.py:207: RuntimeWarning:

invalid value encountered in divide

